# FHIRscan — DIZ-Vergleichsanalyse

Vergleicht die Profiling-Outputs von bis zu **6 Datenintegrationszentren (DIZ)**.  
Jedes DIZ entspricht einem Ordner, den `main.py` erzeugt hat.

**Struktur:**
1. Datensatz-Übersicht (Ressourcenzahlen)
2. Vollständigkeitsanalyse (Presence Rates) — Heatmaps pro Ressourcentyp
3. Datenqualität (Patient/Encounter-Linkage)
4. Kardinalitätsvergleich
5. Größte Abweichungen zwischen DIZ
6. Wertverteilung für kodierte Felder

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.3f}'.format)

## ⚙️ Konfiguration

Trage hier die **Output-Ordner** der 6 DIZ ein (die timestamped Unterordner von `Output/`).  
Nicht verwendete Einträge einfach auskommentieren oder leer lassen.

In [ ]:
# ── Konfiguration: Namen und Pfade der DIZ ────────────────────────────────
# Schlüssel = Anzeigename des DIZ  │  Wert = Pfad zum Output-Ordner

BASE = Path("Output")   # Basisverzeichnis — relativ zum Notebook

DIZ = {
    "DIZ-A": BASE / "20260410_16_44",   # ← Pfad anpassen
    "DIZ-B": BASE / "20260410_16_34",   # ← Pfad anpassen
    "DIZ-C": BASE / "20260410_15_16",   # ← Pfad anpassen
    # "DIZ-D": BASE / "...",
    # "DIZ-E": BASE / "...",
    # "DIZ-F": BASE / "...",
}

# Farben pro DIZ (automatisch erweitert wenn mehr als 6)
DIZ_COLORS = px.colors.qualitative.Bold
COLOR_MAP = {name: DIZ_COLORS[i % len(DIZ_COLORS)] for i, name in enumerate(DIZ)}

print(f"Konfigurierte DIZ: {list(DIZ.keys())}")

## 📥 Daten laden

In [ ]:
def load_csv_safe(path: Path, diz_name: str) -> pd.DataFrame:
    """Lädt eine CSV, gibt leeren DataFrame zurück wenn Datei fehlt."""
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["diz"] = diz_name
    return df


def load_fields(folder: Path, diz_name: str) -> pd.DataFrame:
    """Lädt alle *_fields.csv aus einem Ordner und kombiniert sie."""
    dfs = []
    for csv_file in sorted(folder.glob("*_fields.csv")):
        resource_type = csv_file.stem.replace("_fields", "")
        df = pd.read_csv(csv_file)
        df["diz"] = diz_name
        df["resource_type"] = resource_type
        # Typkonvertierungen
        for col in ["presence_rate", "missing_rate"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def parse_top_values(tv_str) -> dict:
    """Parst den top_values JSON-String zu einem {wert: count}-Dict."""
    if pd.isna(tv_str) or tv_str == "":
        return {}
    try:
        pairs = json.loads(tv_str)
        return {str(k): int(v) for k, v in pairs}
    except Exception:
        return {}


# ── Alle Daten laden ──────────────────────────────────────────────────────
summary_frames, quality_frames, card_frames, fields_frames = [], [], [], []
missing_folders = []

for diz_name, folder in DIZ.items():
    if not folder.exists():
        missing_folders.append((diz_name, str(folder)))
        print(f"⚠️  Ordner nicht gefunden: {diz_name} → {folder}")
        continue

    summary_frames.append(load_csv_safe(folder / "_summary.csv", diz_name))
    quality_frames.append(load_csv_safe(folder / "_data_quality.csv", diz_name))
    card_frames.append(load_csv_safe(folder / "_cardinality.csv", diz_name))
    fields_frames.append(load_fields(folder, diz_name))

df_summary  = pd.concat(summary_frames,  ignore_index=True) if summary_frames  else pd.DataFrame()
df_quality  = pd.concat(quality_frames,  ignore_index=True) if quality_frames  else pd.DataFrame()
df_card     = pd.concat(card_frames,     ignore_index=True) if card_frames     else pd.DataFrame()
df_fields   = pd.concat(fields_frames,   ignore_index=True) if fields_frames   else pd.DataFrame()

# Numerische Kardinaliätsspalten
for col in ["coverage_rate", "mean", "median", "std_dev", "min", "max", "p25", "p75", "p90", "p99"]:
    if col in df_card.columns:
        df_card[col] = pd.to_numeric(df_card[col], errors="coerce")

LOADED_DIZ = [d for d in DIZ if d not in [m[0] for m in missing_folders]]
ALL_RT = sorted(df_summary["resource_type"].unique()) if not df_summary.empty else []

print(f"\n✅ Geladen: {len(LOADED_DIZ)} DIZ, {len(ALL_RT)} Ressourcentypen")
print(f"   Ressourcentypen: {ALL_RT}")
print(f"   Felder gesamt:   {len(df_fields):,} Einträge")

---
## 1. Datensatz-Übersicht

In [ ]:
# ── 1a. Ressourcenzahlen pro DIZ ─────────────────────────────────────────
fig = px.bar(
    df_summary,
    x="resource_type", y="total_resources", color="diz",
    barmode="group",
    color_discrete_map=COLOR_MAP,
    title="Ressourcenzahlen je Ressourcentyp und DIZ",
    labels={"total_resources": "Anzahl Ressourcen", "resource_type": "Ressourcentyp", "diz": "DIZ"},
    height=450
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

In [ ]:
# ── 1b. Feldstruktur-Übersicht ────────────────────────────────────────────
# Zeigt: Felder immer vorhanden vs. manchmal vs. Typinkonsistenz
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Felder immer vorhanden", "Felder manchmal vorhanden", "Typinkonsistenzen"],
    shared_yaxes=False
)

metrics = [
    ("fields_always_present", 1),
    ("fields_sometimes_present", 2),
    ("fields_with_type_inconsistency", 3),
]

for col_name, col_idx in metrics:
    if col_name not in df_summary.columns:
        continue
    for diz_name in LOADED_DIZ:
        sub = df_summary[df_summary["diz"] == diz_name]
        fig.add_trace(
            go.Bar(
                name=diz_name, x=sub["resource_type"], y=sub[col_name],
                marker_color=COLOR_MAP[diz_name],
                showlegend=(col_idx == 1)
            ),
            row=1, col=col_idx
        )

fig.update_layout(
    barmode="group", height=420,
    title_text="Feldstruktur je Ressourcentyp und DIZ",
    legend_title="DIZ"
)
fig.show()

---
## 2. Vollständigkeitsanalyse (Presence Rates)

Heatmap: Zeilen = Felder, Spalten = DIZ, Farbe = `presence_rate` (0 = nie vorhanden, 1 = immer vorhanden).  
Nur skalare Felder werden angezeigt (keine reinen Objekt-/Array-Knoten).

In [ ]:
def presence_heatmap(resource_type: str, min_presence_any_diz: float = 0.0):
    """
    Zeichnet eine Presence-Rate-Heatmap für einen Ressourcentyp.
    min_presence_any_diz: Felder werden nur angezeigt wenn mindestens ein DIZ
                           diesen Wert überschreitet (filtert reine Nullzeilen).
    """
    sub = df_fields[
        (df_fields["resource_type"] == resource_type) &
        # Nur skalare Felder: kein reines object/array ohne weiteren Typ
        (~df_fields["detected_python_types"].str.fullmatch(r"object|array", na=False))
    ].copy()

    if sub.empty:
        print(f"Keine skalaren Felder für {resource_type} gefunden.")
        return

    pivot = sub.pivot_table(
        index="field_path", columns="diz",
        values="presence_rate", aggfunc="first"
    ).reindex(columns=LOADED_DIZ)

    # Filter: nur Felder die in mindestens einem DIZ > min_presence_any_diz sind
    pivot = pivot[pivot.max(axis=1) > min_presence_any_diz]

    if pivot.empty:
        print(f"Keine Felder über Schwellwert {min_presence_any_diz} für {resource_type}.")
        return

    # Sortierung: nach mittlerer Presence Rate (absteigend)
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    height = max(400, len(pivot) * 22 + 100)

    fig = px.imshow(
        pivot,
        color_continuous_scale="RdYlGn",
        zmin=0, zmax=1,
        aspect="auto",
        title=f"Presence Rate — {resource_type}  ({len(pivot)} Felder)",
        labels={"color": "Presence Rate"},
        height=height
    )
    fig.update_xaxes(side="top")
    fig.update_layout(coloraxis_colorbar_title="Rate")
    fig.show()


# ── Heatmap für alle Ressourcentypen ─────────────────────────────────────
for rt in ALL_RT:
    presence_heatmap(rt, min_presence_any_diz=0.01)

In [ ]:
# ── 2b. Abweichungs-Heatmap: Differenz von DIZ-Wert zum DIZ-Mittel ───────
# Grün = DIZ hat höhere Presence Rate als Durchschnitt, Rot = niedriger

def delta_heatmap(resource_type: str):
    sub = df_fields[
        (df_fields["resource_type"] == resource_type) &
        (~df_fields["detected_python_types"].str.fullmatch(r"object|array", na=False))
    ].copy()

    if sub.empty:
        return

    pivot = sub.pivot_table(
        index="field_path", columns="diz",
        values="presence_rate", aggfunc="first"
    ).reindex(columns=LOADED_DIZ)

    # Nur Felder mit Varianz zwischen DIZ
    row_std = pivot.std(axis=1)
    pivot = pivot[row_std > 0.01]
    if pivot.empty:
        print(f"{resource_type}: Keine signifikanten Abweichungen zwischen DIZ.")
        return

    delta = pivot.sub(pivot.mean(axis=1), axis=0)
    delta = delta.loc[pivot.std(axis=1).sort_values(ascending=False).index]

    height = max(400, len(delta) * 22 + 100)

    fig = px.imshow(
        delta,
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="auto",
        title=f"Abweichung vom DIZ-Mittel — {resource_type}  ({len(delta)} Felder mit Varianz)",
        labels={"color": "Δ Presence Rate"},
        height=height
    )
    fig.update_xaxes(side="top")
    fig.show()


for rt in ALL_RT:
    delta_heatmap(rt)

---
## 3. Datenqualität — Patient & Encounter Linkage

In [ ]:
if not df_quality.empty:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Patient Linkage Rate", "Encounter Linkage Rate"]
    )

    for diz_name in LOADED_DIZ:
        sub = df_quality[df_quality["diz"] == diz_name]
        show_legend = (diz_name == LOADED_DIZ[0])

        fig.add_trace(go.Bar(
            name=diz_name, x=sub["resource_type"], y=sub["patient_linkage_rate"],
            marker_color=COLOR_MAP[diz_name], showlegend=True
        ), row=1, col=1)

        fig.add_trace(go.Bar(
            name=diz_name, x=sub["resource_type"], y=sub["encounter_linkage_rate"],
            marker_color=COLOR_MAP[diz_name], showlegend=False
        ), row=1, col=2)

    fig.update_yaxes(range=[0, 1.05], tickformat=".0%")
    fig.update_layout(
        barmode="group", height=430,
        title_text="Datenqualität: Verlinkungsraten je Ressourcentyp und DIZ",
        legend_title="DIZ"
    )
    fig.show()

    # Tabelle: Wer hat die niedrigste Patient-Linkage-Rate?
    print("\n⚠️  Niedrige Patient-Linkage-Raten (< 0.8):")
    low = df_quality[df_quality["patient_linkage_rate"] < 0.8][["diz", "resource_type", "patient_linkage_rate", "total_resources"]]
    if low.empty:
        print("   Keine gefunden.")
    else:
        display(low.sort_values("patient_linkage_rate"))
else:
    print("Keine _data_quality.csv Daten geladen.")

---
## 4. Kardinalitätsvergleich

Wie viele Ressourcen eines Typs hat ein Patient/Encounter durchschnittlich — im Vergleich zwischen DIZ?

In [ ]:
if not df_card.empty:
    for anchor in df_card["anchor_type"].unique():
        sub = df_card[df_card["anchor_type"] == anchor].copy()

        anchor_label = "Patient" if "patient" in anchor else "Encounter"

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[f"Mittlere Anzahl je {anchor_label}", f"Coverage Rate (≥1 Ressource)"]
        )

        for diz_name in LOADED_DIZ:
            d = sub[sub["diz"] == diz_name]
            # Mean mit Fehlerbalken (std_dev)
            fig.add_trace(go.Bar(
                name=diz_name, x=d["resource_type"], y=d["mean"],
                error_y=dict(type="data", array=d["std_dev"].fillna(0).tolist()),
                marker_color=COLOR_MAP[diz_name],
                showlegend=True
            ), row=1, col=1)

            fig.add_trace(go.Bar(
                name=diz_name, x=d["resource_type"], y=d["coverage_rate"],
                marker_color=COLOR_MAP[diz_name],
                showlegend=False
            ), row=1, col=2)

        fig.update_layout(
            barmode="group", height=430,
            title_text=f"Kardinalität: {anchor}",
            legend_title="DIZ"
        )
        fig.update_yaxes(tickformat=".0%", row=1, col=2)
        fig.show()

    # Detailtabelle: Perzentile im Vergleich
    print("\nKardinalitäts-Detailtabelle (per_patient):")
    detail_cols = ["diz", "resource_type", "total_resources", "min", "mean", "median", "max", "p90", "p99", "coverage_rate"]
    available = [c for c in detail_cols if c in df_card.columns]
    display(
        df_card[df_card["anchor_type"] == "per_patient"][available]
        .sort_values(["resource_type", "diz"])
        .reset_index(drop=True)
    )
else:
    print("Keine _cardinality.csv Daten geladen.")

---
## 5. Größte Abweichungen zwischen DIZ

Welche Felder unterscheiden sich am stärksten in ihrer Presence Rate zwischen den DIZ?

In [ ]:
# Parameter anpassen:
TOP_N_FIELDS = 25   # Wie viele Felder je Ressourcentyp anzeigen
MIN_PRESENCE = 0.01  # Felder ignorieren die überall < 1% vorhanden sind

if not df_fields.empty and len(LOADED_DIZ) > 1:
    scalar_fields = df_fields[
        ~df_fields["detected_python_types"].str.fullmatch(r"object|array", na=False)
    ].copy()

    results = []
    for rt in ALL_RT:
        sub = scalar_fields[scalar_fields["resource_type"] == rt]
        pivot = sub.pivot_table(
            index="field_path", columns="diz",
            values="presence_rate", aggfunc="first"
        ).reindex(columns=LOADED_DIZ)

        pivot = pivot[pivot.max(axis=1) > MIN_PRESENCE]
        if pivot.empty:
            continue

        pivot["std"] = pivot[LOADED_DIZ].std(axis=1)
        pivot["range"] = pivot[LOADED_DIZ].max(axis=1) - pivot[LOADED_DIZ].min(axis=1)
        pivot["mean"] = pivot[LOADED_DIZ].mean(axis=1)
        pivot["resource_type"] = rt
        pivot = pivot.reset_index()
        results.append(pivot)

    if results:
        df_divergence = pd.concat(results, ignore_index=True)
        df_divergence = df_divergence.sort_values("range", ascending=False)

        # ── Top-N global ──────────────────────────────────────────────────
        top_global = df_divergence.head(TOP_N_FIELDS)

        fig = px.bar(
            top_global,
            x="range", y="field_path",
            color="resource_type",
            orientation="h",
            title=f"Top {TOP_N_FIELDS} Felder mit größter Presence-Rate-Spannweite zwischen DIZ",
            labels={"range": "Max − Min Presence Rate", "field_path": "Feldpfad"},
            height=max(500, TOP_N_FIELDS * 26 + 120)
        )
        fig.update_layout(yaxis=dict(autorange="reversed"))
        fig.show()

        # ── Top-N pro Ressourcentyp ───────────────────────────────────────
        print("\nTop-Abweichungen je Ressourcentyp:")
        display_cols = ["resource_type", "field_path", "mean", "std", "range"] + LOADED_DIZ
        available = [c for c in display_cols if c in df_divergence.columns]
        display(
            df_divergence.groupby("resource_type")
            .head(5)[available]
            .sort_values(["resource_type", "range"], ascending=[True, False])
            .reset_index(drop=True)
            .style.format({c: "{:.3f}" for c in ["mean", "std", "range"] + LOADED_DIZ if c in df_divergence.columns})
            .background_gradient(subset=["range"], cmap="YlOrRd")
        )
else:
    print("Mindestens 2 DIZ erforderlich für Divergenzanalyse.")

---
## 6. Wertverteilung für kodierte Felder

Vergleicht die Verteilung der häufigsten Werte (`top_values`) für ausgewählte Felder zwischen den DIZ.  
Nützlich z.B. für: Diagnose-Codes, LOINC-Codes, Status-Werte.

In [ ]:
# ── Parameter: Felder für die Wertverteilung anzeigen ─────────────────────
# Kommentiere Felder aus die in deinen Daten nicht vorhanden sind.
VALUE_FIELDS_OF_INTEREST = [
    # Passe diese Liste an deine tatsächlichen Feldpfade an:
    "Observation.code.coding[].code",
    "Condition.code.coding[].code",
    "Encounter.class",
    "Observation.status",
    "Condition.clinicalStatus.coding[].code",
    "Procedure.status",
]
TOP_VALUES_SHOW = 15   # Wie viele Werte pro Feld anzeigen

# ─────────────────────────────────────────────────────────────────────────
def plot_value_distribution(field_path: str):
    sub = df_fields[df_fields["field_path"] == field_path]
    if sub.empty:
        print(f"Feld '{field_path}' nicht in den Daten gefunden.")
        return

    all_diz_data = []
    for _, row in sub.iterrows():
        tv = parse_top_values(row.get("top_values", ""))
        for value, count in list(tv.items())[:TOP_VALUES_SHOW]:
            all_diz_data.append({
                "diz": row["diz"],
                "value": value,
                "count": count,
                "total": row.get("value_count", 1)
            })

    if not all_diz_data:
        print(f"Keine top_values Daten für '{field_path}'.")
        return

    df_tv = pd.DataFrame(all_diz_data)
    df_tv["rate"] = df_tv["count"] / df_tv["total"]

    # Top-Werte global (nach Gesamtvorkommen sortieren)
    top_values_global = (
        df_tv.groupby("value")["count"].sum()
        .sort_values(ascending=False)
        .head(TOP_VALUES_SHOW)
        .index.tolist()
    )
    df_tv = df_tv[df_tv["value"].isin(top_values_global)]

    fig = px.bar(
        df_tv,
        x="value", y="rate", color="diz",
        barmode="group",
        color_discrete_map=COLOR_MAP,
        title=f"Wertverteilung: {field_path}",
        labels={"rate": "Anteil an Feldwerten", "value": "Wert", "diz": "DIZ"},
        height=420
    )
    fig.update_yaxes(tickformat=".1%")
    fig.update_xaxes(tickangle=-30)
    fig.show()


for field_path in VALUE_FIELDS_OF_INTEREST:
    plot_value_distribution(field_path)

In [ ]:
# ── 6b. Automatisch die Felder mit den meisten distinkt-Werten finden ─────
# (als Ausgangspunkt für manuelle Auswahl oben)

if not df_fields.empty:
    coded_candidates = df_fields[
        (df_fields["detected_fhir_type"].isin(["code", "CodeableConcept", "Coding", "uri"])) &
        (df_fields["presence_rate"] > 0.5) &
        (~df_fields["detected_python_types"].str.fullmatch(r"object|array", na=False)) &
        (df_fields["unique_count"] > 1) &
        (df_fields["unique_count"] < 500)   # Zu viele Werte = keine sinnvolle Verteilung
    ].copy()

    if not coded_candidates.empty:
        print("Vorschläge für VALUE_FIELDS_OF_INTEREST (oft vorhandene kodierte Felder):")
        display(
            coded_candidates
            .groupby(["resource_type", "field_path"])["unique_count"]
            .mean()
            .sort_values(ascending=False)
            .head(20)
            .reset_index()
            .rename(columns={"unique_count": "Ø unique_count"})
        )

---
## 7. Export

Zusammengefasste Divergenztabelle als CSV speichern.

In [ ]:
EXPORT_PATH = Path("diz_comparison_results")
EXPORT_PATH.mkdir(exist_ok=True)

exported = []

# Presence-Rate-Pivot (alle Felder)
if not df_fields.empty:
    scalar_all = df_fields[
        ~df_fields["detected_python_types"].str.fullmatch(r"object|array", na=False)
    ]
    pivot_all = scalar_all.pivot_table(
        index=["resource_type", "field_path"], columns="diz",
        values="presence_rate", aggfunc="first"
    ).reindex(columns=LOADED_DIZ).reset_index()
    pivot_all.columns.name = None
    out = EXPORT_PATH / "presence_rate_all_fields.csv"
    pivot_all.to_csv(out, index=False)
    exported.append(out)

# Divergenztabelle
if 'df_divergence' in dir() and not df_divergence.empty:
    out = EXPORT_PATH / "field_divergence.csv"
    df_divergence.to_csv(out, index=False)
    exported.append(out)

# Data Quality
if not df_quality.empty:
    out = EXPORT_PATH / "data_quality_comparison.csv"
    df_quality.to_csv(out, index=False)
    exported.append(out)

# Kardinality
if not df_card.empty:
    out = EXPORT_PATH / "cardinality_comparison.csv"
    df_card.to_csv(out, index=False)
    exported.append(out)

print("Exportiert:")
for p in exported:
    print(f"  {p}")